# 📘 Lecture 4: Hands-On Document Q&A Agent with LangChain

In this notebook, we’ll build a fully functional **Document Q&A AI Agent** using LangChain, OpenAI, and FAISS.

You'll learn to:
- Load and index a `.txt` file
- Use semantic search with embeddings
- Build a Retrieval-Augmented Generation (RAG) pipeline
- Ask multiple questions and retrieve answers with sources

Let's get started!

## 🔧 Step 1: Import Required Libraries

In [ ]:
from pathlib import Path
import sys

def _find_root():
    hints = [
        Path("data") / "udemy" / "courses" / "ai_agents_bootcamp" / "course_repo",
        Path("course_repo"),
    ]
    def ok(p: Path) -> bool:
        return (p / "src" / "llm.py").is_file() or (p / "Section_5_Autonomous_Workflows").is_dir()
    cur = Path.cwd().resolve()
    for p in [cur, *cur.parents]:
        if ok(p):
            return p
        for h in hints:
            cand = p / h
            if ok(cand):
                return cand.resolve()
    raise FileNotFoundError("Open this notebook from the AIAgentsBootcamp folder, or set cwd to that repo.")

_ROOT = _find_root()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))
from src.llm import REPO_ROOT, get_agent, get_embeddings, get_llm
llm = get_llm()


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from dotenv import load_dotenv
import os


## 🔐 Step 2: Load Environment Variables and Set Up LLM

In [ ]:
from pathlib import Path
import sys

def _find_root():
    hints = [
        Path("data") / "udemy" / "courses" / "ai_agents_bootcamp" / "course_repo",
        Path("course_repo"),
    ]
    def ok(p: Path) -> bool:
        return (p / "src" / "llm.py").is_file() or (p / "Section_5_Autonomous_Workflows").is_dir()
    cur = Path.cwd().resolve()
    for p in [cur, *cur.parents]:
        if ok(p):
            return p
        for h in hints:
            cand = p / h
            if ok(cand):
                return cand.resolve()
    raise FileNotFoundError("Open this notebook from the AIAgentsBootcamp folder, or set cwd to that repo.")

_ROOT = _find_root()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))
from src.llm import REPO_ROOT, get_agent, get_embeddings, get_llm
llm = get_llm()


## 📄 Step 3: Load the Sample FAQ Document and Split into Chunks

In [10]:
loader = TextLoader(str(REPO_ROOT / "Section_6_Real_World_RAG_Engineering" / "sample_faq.txt"), encoding="utf-8")
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

## 🧠 Step 4: Convert Text to Vectors and Index with FAISS

In [11]:
vectorstore = FAISS.from_documents(docs, embeddings)

## 🛠️ Step 5: Setup Retrieval-Augmented QA Chain

In [12]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 2}),
    return_source_documents=True
)

## ❓ Step 6: Ask Multiple Questions and Display Answers with Sources

In [13]:
questions = [
    "What is the refund policy?",
    "How long does shipping usually take?",
    "Can customers cancel their subscription anytime?",
    "What are the benefits of the premium plan?"
]

for q in questions:
    response = qa_chain.invoke({"query": q})
    print(f"\n❓ Question: {q}")
    print(f"✅ Answer: {response['result']}")
    print("📚 Source Document Snippets:")
    for doc in response["source_documents"]:
        print(" -", doc.page_content[:150], "...")


❓ Question: What is the refund policy?
✅ Answer: The refund policy offers a 30-day money-back guarantee on all plans. If you're not satisfied, you can contact support within 30 days for a full refund.
📚 Source Document Snippets:
 - Refund Policy:
We offer a 30-day money-back guarantee on all our plans. If you're not satisfied, contact support within 30 days for a full refund.

Sh ...
 - Subscription Cancellation:
Customers can cancel their subscription anytime through the account settings page. No additional charges will apply after c ...

❓ Question: How long does shipping usually take?
✅ Answer: Shipping typically takes between 5-7 business days, depending on your location. International orders may take longer.
📚 Source Document Snippets:
 - Refund Policy:
We offer a 30-day money-back guarantee on all our plans. If you're not satisfied, contact support within 30 days for a full refund.

Sh ...
 - Subscription Cancellation:
Customers can cancel their subscription anytime through the 

## ✅ Wrap-Up

You've now built a real document Q&A system using:
- GPT-4 with LangChain
- FAISS for fast vector retrieval
- TextLoader and TextSplitter for document prep

This is a powerful template for internal knowledge bots, helpdesk assistants, and intelligent agents.

Next, you’ll learn to enhance this pipeline using tools and advanced workflows.